# Precompute session-level trial metadata directly from ONE

Fetches `choice`, `feedbackType` (outcome), `probabilityLeft` (block), `goCueTrigger_times`,
contrast (signed and unsigned), `reaction` and `elongation` times, straight from ONE for every
LDA-tracked session (the same session set used by `firing_rate_computation.ipynb`), plus the
GLM-HMM engagement state (`p_state1`/`dominant_state`), and saves one consolidated file.

This replaces dependence on `4_mice/all_trials_04-05-2026`, which was generated by the now-deprecated
`0_syllable_trial_type.ipynb` pipeline (predates the no-go/choice fixes made to `segmentation_functions.py`
and silently drops every no-go trial). Fetching live from ONE avoids inheriting any of that pipeline's bugs.

`goCueTrigger_times` is included specifically so that consuming notebooks (e.g. `noise_corr_psth_conditions.ipynb`)
can validate the `trial_id` join against each firing-rate file's own `event_time` column before trusting it,
rather than assuming `trial_id` alignment silently.

`reaction`/`elongation` use the same definitions as `segmentation_functions.py`'s `prepro()`:
- `reaction = firstMovement_times - goCue_times`
- `elongation = goCue_times - quiescencePeriod - intervals_0` (Pre-quiescence duration: time from
  trial start until quiescence begins, which varies because the animal has to keep re-settling
  into stillness before the quiescence timer can complete)

`contrast`/`signed_contrast` are derived from `contrastLeft`/`contrastRight` (NaN-filled to 0):
`contrast = |contrastLeft| + |contrastRight|` (magnitude), `signed_contrast = contrastRight - contrastLeft`
(negative=left, positive=right, matching `GLM-HMM/merged_behavioral_and_states.pqt`'s convention).
`firing_rate_computation.ipynb` derives its own `condition` column from these fields for its own
purposes, so there's some redundancy there, but it's cheap since `contrastLeft`/`contrastRight` are
already part of the same `trials` object fetched here.

choice/feedback string convention (matches `segmentation_functions.py`'s `align_bin_design_matrix`,
verified against a known trial - see notebook discussion):
- choice:   1 -> 'right', -1 -> 'left', 0 -> 'no_go'
- feedback: 1 -> 'correct', -1 -> 'incorrect'

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
from one.api import ONE
import brainwidemap

one = ONE()
prefix = '/home/ines/repositories/'
prefix = '/Users/ineslaranjeira/Documents/Repositories/'


## Load the LDA session list (same source as `firing_rate_computation.ipynb`)

In [1]:
bwm_df = brainwidemap.bwm_query()
n_sessions = bwm_df['eid'].unique().shape[0]
n_insertions = bwm_df['pid'].unique().shape[0]
print(f'{n_sessions} sessions with {n_insertions} individual neuropixel recordings')

data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
lda = pd.read_pickle(data_path + 'mouse_LDA_5_bins_cut19-06-2026')  # local Mac copy; remote uses mouse_LDA_5_bins_cut06-07-2026
lda = lda.rename(columns={0: 'lda_1'})

lda_eid = lda.loc[lda['session'].isin(list(bwm_df.eid)), 'session']
sessions = sorted(lda_eid.unique())
print(f'{len(sessions)} unique LDA-matched sessions to fetch')


KeyboardInterrupt



## Fetch trial metadata per session

In [3]:
CHOICE_MAP = {1.0: 'right', -1.0: 'left', 0.0: 'no_go'}
FEEDBACK_MAP = {1.0: 'correct', -1.0: 'incorrect'}

records = []
failed_sessions = {}

for i, session in enumerate(sessions):
    try:
        trials = one.load_object(session, 'trials', collection='alf')
    except Exception as e:
        print(f'  Could not fetch trials for {session}: {e}')
        failed_sessions[session] = str(e)
        continue

    n = len(trials['choice'])
    contrast_left = np.nan_to_num(np.asarray(trials['contrastLeft']))
    contrast_right = np.nan_to_num(np.asarray(trials['contrastRight']))
    intervals_0 = np.asarray(trials['intervals'])[:, 0]

    df = pd.DataFrame({
        'session': session,
        'trial_id': np.arange(n),
        'choice': pd.Series(trials['choice']).map(CHOICE_MAP),
        'feedback': pd.Series(trials['feedbackType']).map(FEEDBACK_MAP),
        'block': np.asarray(trials['probabilityLeft']),
        'goCueTrigger_times': np.asarray(trials['goCueTrigger_times']),
        # unsigned magnitude and signed (negative=left, positive=right) contrast
        'contrast': contrast_left + contrast_right,
        'signed_contrast': contrast_right - contrast_left,
        # reaction and (quiescence) elongation, same definitions as segmentation_functions.py's prepro()
        'reaction': np.asarray(trials['firstMovement_times']) - np.asarray(trials['goCue_times']),
        'elongation': np.asarray(trials['goCue_times']) - np.asarray(trials['quiescencePeriod']) - intervals_0,
    })
    records.append(df)
    if (i + 1) % 25 == 0:
        print(f'  {i+1}/{len(sessions)} sessions fetched...')

session_trial_meta = pd.concat(records, ignore_index=True)
print(f'\nFetched {len(session_trial_meta)} trial rows across {session_trial_meta["session"].nunique()} sessions')
if failed_sessions:
    print(f'{len(failed_sessions)} sessions failed to fetch: {list(failed_sessions.keys())}')

/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


...

/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)



Fetched 154375 trial rows across 244 sessions


## Add GLM-HMM engagement state

`p_state1`/`dominant_state` ('state1'=engaged, 'state2'=disengaged, thresholded at p_state1>=0.5)
from `GLM-HMM/merged_behavioral_and_states.pqt`. Joined on (`session`, `trial_id`), where
`trial_id` for the GLM-HMM file is reconstructed as row-order-within-session
(`groupby('eid').cumcount()`) - validated repeatedly elsewhere in this repo (`load_states.ipynb`,
`syllables_vs_engagement.ipynb`) that this row order matches the same ONE trial order used here,
via exact `feedbackType`/`contrastLeft`/`contrastRight` matches. Not every session in
`sessions` necessarily has a GLM-HMM fit, so this is a left join (unmatched trials get NaN).

In [4]:
states_df = pd.read_parquet(prefix + 'representation_learning_variability/paper-individuality/GLM-HMM/merged_behavioral_and_states.pqt')
states_df['dominant_state'] = np.where(states_df['p_state1'] >= 0.5, 'state1', 'state2')
states_df['trial_id'] = states_df.groupby('eid').cumcount()

session_trial_meta = session_trial_meta.merge(
    states_df[['eid', 'trial_id', 'p_state1', 'dominant_state']].rename(columns={'eid': 'session'}),
    on=['session', 'trial_id'], how='left',
)

n_matched_trials = session_trial_meta['p_state1'].notna().sum()
n_matched_sessions = session_trial_meta.dropna(subset=['p_state1'])['session'].nunique()
print(f'Trials with GLM-HMM engagement state: {n_matched_trials} / {len(session_trial_meta)}')
print(f'Sessions with GLM-HMM engagement state: {n_matched_sessions} / {session_trial_meta["session"].nunique()}')

Trials with GLM-HMM engagement state: 154375 / 154375
Sessions with GLM-HMM engagement state: 244 / 244


In [ ]:
mouse_map = lda.drop_duplicates('session')[['session', 'mouse_name']]
session_trial_meta = session_trial_meta.merge(mouse_map, on='session', how='left')
print(f'Trials with mouse_name: {session_trial_meta["mouse_name"].notna().sum()} / {len(session_trial_meta)}')

## Save

In [5]:
save_path = prefix + 'representation_learning_variability/paper-individuality/4_mice/'
current_date = datetime.now().strftime('%d-%m-%Y')
filename = f'session_trial_meta_{current_date}'
session_trial_meta.to_parquet(save_path + filename, compression='gzip')
print(f'Saved to {save_path + filename}')

Saved to /Users/ineslaranjeira/Documents/Repositories/representation_learning_variability/paper-individuality/4_mice/session_trial_meta_10-07-2026
